# Part 4: Errors and logging

**Goal:** Handle service failures with short waits and useful log messages.

**Time:** 25 minutes. **Run cells top to bottom.**

First, make sure the two small libraries used in the examples are installed. Your local environment usually already has them; Colab may need them.

In [ ]:
import importlib.util
import subprocess
import sys
missing = [pkg for module, pkg in [("requests", "requests"),
           ("dotenv", "python-dotenv")] if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

Find the workshop folder. In Colab, download a fresh copy if needed. This also makes `starter/` imports and file paths work in every notebook.

In [ ]:
import os
from pathlib import Path
import tempfile
root = Path.cwd()
if not (root / "starter").exists() and (root.parent / "starter").exists():
    root = root.parent
if not (root / "starter").exists():
    root = Path(tempfile.mkdtemp()) / "workshop"
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/HruthikKommuru/cwru-fall26-session7.git", str(root)])
os.chdir(root)
sys.path.insert(0, str(root))

Load the tools. A **timeout** limits how long one request waits; a bad HTTP status means the server responded but could not do what we asked.

In [ ]:
import requests
import time
import logging
logging.basicConfig(level=logging.INFO)
log = logging.getLogger("workshop.retries")

Try a server that deliberately returns status 500. The tiny timeout may fire first. Both outcomes are handled; other connection failures also get a clear result.

In [ ]:
url = "https://httpbin.org/status/500"
try:
    r = requests.get(url, timeout=0.001)
    r.raise_for_status()
    result = {"status": "ok"}
except requests.Timeout:
    result = {"error": "timeout"}
except requests.HTTPError:
    result = {"error": "bad status"}
except requests.RequestException:
    result = {"error": "connection unavailable"}
print(result)

A retry gives a temporary failure another chance. **Backoff** increases waits to 1, 2, then 4 seconds. The **cap** of three attempts prevents an endless loop.

In [ ]:
def call_with_retries(call_api):
    for attempt in range(3):
        try:
            return call_api()
        except requests.Timeout:
            time.sleep(2 ** attempt)
    return {"error": "gave up after 3 tries"}

Unlike a quick `print`, a log message has a level and a named source. WARNING means a retry; ERROR means we gave up. `%d` and `%s` insert values.

In [ ]:
n, city = 1, "Cleveland"
log.warning("retry %d for %s", n, city)
log.error("gave up on %s", city)

## Your turn — Lab 4

Import the pretend flaky API and write `retry_flaky(wait_scale=0.1)`. It should
try at most three times, catch `requests.Timeout`, log every retry as WARNING,
and wait `wait_scale * 2 ** attempt` seconds. After three failures, log ERROR
and return `{"error": "gave up after 3 tries"}`. Success returns `{"status": "ok"}`.

Use `wait_scale=1` for real 1s/2s/4s waits; 0.1 gives 0.1s/0.2s/0.4s in class.
We keep a final wait here to match the teaching loop above.

Replace the placeholder with your retry function. Use the same named logger as the example so the check can capture its messages.

In [ ]:
from starter.flaky_api import flaky_api
import random
retry_flaky = None
# TODO: def retry_flaky(wait_scale=0.1): ...
# TODO: three attempts, WARNING on each timeout, increasing waits.
# TODO: ERROR and a clear error dictionary when all attempts fail.

Run your wrapper ten times. A fixed random seed makes this experiment repeatable; count how many runs returned success.

In [ ]:
random.seed(7)
outcomes = [] if retry_flaky is None else [retry_flaky() for _ in range(10)]
successes = sum(item.get("status") == "ok" for item in outcomes)
print(f"Successes: {successes} out of {len(outcomes)}")

**Check yourself:** with this seed, ten runs produce nine successes and one clear failure. The next cell also checks the retry rules directly.

In [ ]:
if retry_flaky is None:
    print("Write retry_flaky, then rerun the ten calls and this check.")
else:
    assert len(outcomes) == 10
    assert successes == 9
    assert all(item in ({"status": "ok"}, {"error": "gave up after 3 tries"}) for item in outcomes)
    print("Nice work! Your ten runs produced stable, clear results.")

This provided helper forces three timeouts without waiting. It checks the 1/2/4 timing plan and log levels without depending on chance.

In [ ]:
from unittest.mock import patch
def check_retry_rules(wrapper):
    with patch("starter.flaky_api.random.random", return_value=0):
        with patch.object(time, "sleep") as sleep, patch.object(log, "warning") as warn:
            with patch.object(log, "error") as error:
                result = wrapper(wait_scale=0.1)
    assert result == {"error": "gave up after 3 tries"}
    assert [call.args[0] for call in sleep.call_args_list] == [0.1, 0.2, 0.4]
    assert warn.call_count == 3 and error.call_count == 1

**Check yourself:** use the helper to verify the failure path too. A good wrapper handles both success and failure predictably.

In [ ]:
if retry_flaky is not None:
    check_retry_rules(retry_flaky)
    print("Nice work! Retry timing and WARNING/ERROR logs passed.")

## What you learned

- Catch specific failures so your program can continue.
- Backoff and an attempt cap keep retries controlled.
- Logs explain retries and final failures.